In [1]:
import carla
import time
client = carla.Client('localhost', 2000)
world = client.get_world()

def find_obj(world, keyword, offset=0):
    env_objs = world.get_environment_objects()
    objs = []

    for env_obj in env_objs:
        if keyword in env_obj.name:
            _env_obj = env_obj
            _env_obj.transform.location.z += offset
            objs.append(_env_obj)

    return objs

spawn_point = find_obj(world, "truck_spawn", 0.5)[0]

truck_bp = world.get_blueprint_library().find("vehicle.daf.dafxf")
trailer_bp = world.get_blueprint_library().find("vehicle.trailer.trailer")

In [2]:
from parking_env_3 import *
env = ParkingLotEnv()

env.reset()
world.get_actor(world.get_actors().filter("vehicle.daf*")[0].id).apply_control(carla.VehicleControl(brake=0.5, reverse=False))

/home/falah/carla/My Code/venv3.10/lib/python3.10/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/falah/carla/My Code/venv3.10/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Destroying 0 actors...
Truck Spawned.
Trailer Spawned and Attached.
Spawned 24 actors (vehicles + sensors).


In [4]:
settings = world.get_settings()
settings.no_rendering_mode = False
world.apply_settings(settings)

79227

In [ ]:
while True:
    time.sleep(1)
    print(env._get_observation()['radar_data'])

In [6]:
print(env._get_observation()['radar_data'])

[[0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


In [ ]:
for sensor in world.get_actors().filter("sensor.other.obstacle*"):
    sensor_transform = sensor.get_transform()

    world.debug.draw_arrow(
        sensor_transform.location,
        sensor_transform.location + sensor_transform.get_forward_vector() * 1.3,
        thickness=0.1,
        arrow_size=0.1,
        color=carla.Color(r=255, g=125, b=0),
        life_time=10
    )

In [3]:
env.destroy_actors()

Destroying 24 actors...


In [ ]:
kill_carla()

: 

In [ ]:
import numpy as np

# Get the trailer parking object
parking_point = find_obj(world, 'trailer_parking')[0]
parking_location = parking_point.transform.location

# Get the trailer actor
trailer = world.get_actors().filter("vehicle.trailer*")[0]
trailer_transform = trailer.get_transform()

# Calculate trailer's back end position
trailer_yaw_rad = np.deg2rad(trailer_transform.rotation.yaw)
trailer_backward = np.array([-np.cos(trailer_yaw_rad), -np.sin(trailer_yaw_rad)])
trailer_bb = trailer.bounding_box
trailer_back_end = np.array([trailer_transform.location.x - 5.0, trailer_transform.location.y]) + trailer_backward * trailer_bb.extent.x

# Draw line from trailer back end to parking spot
world.debug.draw_line(
    carla.Location(x=trailer_back_end[0], y=trailer_back_end[1], z=0.5),
    carla.Location(x=parking_location.x, y=parking_location.y, z=0.5),
    thickness=0.1,
    color=carla.Color(r=0, g=255, b=0),
    life_time=50.0
)